# Meta-Algorithms: Blending Ensemble for Iris Classification

This notebook demonstrates the full workflow for training and evaluating a blending ensemble (a meta-algorithm) on the Iris dataset. Blending combines base learners using a holdout validation set and a meta-learner, offering a simpler alternative to stacking.

## 1. Mathematical Overview

**Blending Algorithm:**
1. Split data: train (60%), blending/validation (20%), test (20%)
2. Train base learners $L_1, L_2, ..., L_k$ on train set
3. Generate meta-features on blending set: $M = [L_1(X_{blend}), L_2(X_{blend}), ..., L_k(X_{blend})]$
4. Train meta-learner $m$ on $(M, y_{blend})$
5. Final prediction: $at{y} = m(M_{test})$

**Key Advantage over Stacking:** Uses fixed holdout instead of k-fold CV, making it faster when computational budget is limited.

## 2. Setup and Data Loading

In [ ]:
# Core libraries
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully.")

## 3. Load and Explore Iris Dataset

In [ ]:
# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Create DataFrame for exploration
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
df['species'] = df['target'].map({i: name for i, name in enumerate(target_names)})

print(f"Dataset shape: {X.shape}")
print(f"\nClasses: {target_names}")
print(f"\nClass distribution:\n{df['species'].value_counts()}")
df.head()

In [ ]:
# Exploratory plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, feature in enumerate(feature_names):
    ax = axes[idx // 2, idx % 2]
    for target, species in enumerate(target_names):
        mask = df['target'] == target
        ax.histogram(df[mask][feature], alpha=0.5, label=species, bins=15)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Train-Validation-Test Split

In [ ]:
# First split: separate test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: separate train and blending sets
X_train, X_blend, y_train, y_blend = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_blend = scaler.transform(X_blend)
X_test = scaler.transform(X_test)

print(f"Train set: {X_train.shape}")
print(f"Blending (validation) set: {X_blend.shape}")
print(f"Test set: {X_test.shape}")

## 5. Train Base Learners

In [ ]:
# Initialize base learners
base_learners = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
}

# Train base learners
print("Training base learners...")
for name, learner in base_learners.items():
    learner.fit(X_train, y_train)
    acc = learner.score(X_test, y_test)
    print(f"{name:20s} - Test Accuracy: {acc:.4f}")

## 6. Generate Meta-Features

In [ ]:
# Generate meta-features (probability predictions) on blending set
print("Generating meta-features...")
meta_features_blend = np.column_stack(
    [learner.predict_proba(X_blend) for learner in base_learners.values()]
)

meta_features_test = np.column_stack(
    [learner.predict_proba(X_test) for learner in base_learners.values()]
)

print(f"Meta-features shape (blending): {meta_features_blend.shape}")
print(f"Meta-features shape (test): {meta_features_test.shape}")
print(f"\nMeta-features are stacked probabilities from {len(base_learners)} base learners × 3 classes")

## 7. Train Meta-Learner

In [ ]:
# Train meta-learner on meta-features
meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner.fit(meta_features_blend, y_blend)

print("Meta-learner trained on blending set meta-features.")
print(f"Meta-learner classes: {meta_learner.classes_}")

## 8. Evaluate Blending Ensemble

In [ ]:
# Make predictions with ensemble
y_pred_ensemble = meta_learner.predict(meta_features_test)
ensemble_accuracy = accuracy_score(y_test, y_pred_ensemble)

# Compare with individual base learners
print("\n=== ACCURACY COMPARISON ===")
print(f"Blending Ensemble: {ensemble_accuracy:.4f}")
print()

for name, learner in base_learners.items():
    y_pred = learner.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name:20s}: {acc:.4f}")

# Calculate improvement
best_base = max([accuracy_score(y_test, learner.predict(X_test)) for learner in base_learners.values()])
improvement = ensemble_accuracy - best_base
print(f"\nImprovement over best base learner: {improvement:.4f} ({improvement*100:.2f}%)")

## 9. Detailed Classification Report

In [ ]:
print("Classification Report for Blending Ensemble:\n")
print(classification_report(y_test, y_pred_ensemble, target_names=target_names))

## 10. Confusion Matrix Visualization

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_ensemble)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Blending Ensemble Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 11. Meta-Feature Analysis

In [ ]:
# Analyze meta-feature importance (coefficients in meta-learner)
coef_df = pd.DataFrame(
    meta_learner.coef_,
    columns=[f"Base {i} - Class {c}" for i in range(len(base_learners)) for c in range(3)],
    index=target_names
)

print("Meta-learner coefficients (feature importance):")
print(coef_df)

# Visualize
plt.figure(figsize=(12, 4))
sns.heatmap(coef_df, annot=True, fmt='.3f', cmap='RdBu_r', center=0)
plt.title('Meta-Learner Feature Importance (Coefficients)')
plt.tight_layout()
plt.show()

## 12. Key Observations and Insights

- **Blending is faster than Stacking:** Uses single holdout validation set instead of k-fold CV
- **Meta-learner learns weights:** Logistic regression learns optimal combination of base learner predictions
- **Ensemble vs. Base:** Often outperforms best base learner due to diversity
- **Trade-offs:** Holdout validation uses less data for meta-learner training but avoids hyperparameter tuning overhead